# 2. Excite the Track

This second part of the user guide builds on the track model created in the [first part](create_track.ipynb). In **Rolland**, analyzing a track's dynamic behavior follows three steps: an excitation is defined, the simulation is then run and the excitation is applied to the track, and the results are finally evaluated in a postprocessing step. Several options are available for the excitation and the simulation, as shown in the following sections. The postprocessing is covered in the [third part](evaluate_results.ipynb).

In [1]:
%%capture
# Setup: recreate rail, track and domain from part 1 (hidden in docs)
%run create_track.ipynb

## 2.1 Defining the Excitation

To analyze the dynamic behavior of the track, an external force must be applied to the rail. Just like the track components, the excitation is created as an independent instance and is later passed, together with the domain setup, to the simulation. For a **stationary excitation**, the force acts at a fixed position along the track and is applied to the rail head, either vertically (Pos. 1) or laterally (Pos. 2), as shown in the figure below. In Rolland, a stationary excitation is realized by the `GaussianImpulse` class.

**Gaussian Impulse Assumptions & Parameters:**
*   The force time history follows a Gaussian impulse according to Stampka (2022), $F(t) = a \, \frac{t - 4\sigma}{\sigma^2} \, e^{-(t - 4\sigma)^2 / \sigma^2}$. The time shift of $4\sigma$ ensures that the force starts smoothly from zero.
*   The pulse parameter `sigma` controls the pulse duration and thus the excited frequency range: a shorter pulse excites higher frequencies. The parameter `a` scales the force amplitude.
*   The excitation position `x_excit` is measured from the left end of the track model, including the boundary domain. The default value of 50 m places the excitation in the middle of the default 100 m track.
*   The force direction `force_dir` is either `'vertical'` ($F_z$, Pos. 1) or `'lateral'` ($F_y$, Pos. 2).
*   Since the force is applied at the rail head but the track model is evaluated at the rail centroid $C_r$, eccentricities `z_e` and `y_e` can be specified. An eccentric force additionally induces a torsional moment about the rail axis. In the postprocessing, the same eccentricities are used to evaluate the response directly at the point of excitation.
*   The eccentricities are coordinates relative to the rail centroid $C_r$ in the coordinate system shown in the figure. Since the $z$-axis points downward, a force applied above $C_r$ has a **negative** `z_e`. For a lateral force at Pos. 2 on a `60E1` rail, for example, `z_e = -71e-3` m. Only the eccentricity that matches the force direction is used: `y_e` for a vertical force and `z_e` for a lateral force.


![Definition of the excitation positions and forces.](../images/definition_of_excitation.png)

In [2]:
from rolland.excitation import GaussianImpulse

# Define a vertical Gaussian impulse at the rail head
exc_vert = GaussianImpulse(
    x_excit=30.3,  # Excitation position (mid-span) [m]
    force_dir="vertical",  # Vertical force F_z (Pos. 1)
    sigma=0.7e-4,  # Pulse duration [s]
    a=0.5e2,  # Pulse amplitude [N]
    y_e=0.0,  # Y-eccentricity [m]
)

## 2.2 Running the Simulation

With the track, the computational domain and the excitation defined, the simulation can be run. In **Rolland**, this is handled by the `Deflection` class. The domain setup and the excitation are passed to it, the excitation is injected into the track, and the finite-difference time-domain update equations are solved step by step. The simulation is executed directly when the `Deflection` instance is created, so no separate run command is required. The results are the vertical deflection `u_z_obs`, the lateral deflection `u_y_obs` and the torsional rotation `phi_x_obs` of the rail centroid.

### Simulation Options in Rolland

Since storing the complete wavefield in space and time quickly becomes memory-intensive, the `store` parameter controls where the track response is recorded:
* `store='excit'`: Records the response only at the excitation position `x_excit`. This is the most efficient option and is sufficient to evaluate the point mobility at the excitation point.
* `store='observe'`: Records the response at one or more user-defined positions `obs_pos` (a single value or a list, in m). This is used to evaluate the transfer mobility at a certain distance from the excitation, e.g. `obs_pos=40.3` for a distance of 10 m from an excitation at 30.3 m.
* `store='full'`: Records the response at every grid point along the entire track. This option requires the most memory, but it is needed for spatially resolved evaluations such as the track decay rate.

### Defaults and Time Settings
* **Sub-sampling (`skip = 1`)**: By default, every time step is stored. With `skip`, only every `skip`-th time step is saved, which reduces the size of the stored results. For `store='full'`, the sub-sampling is already applied during the simulation and therefore also reduces the memory demand.
* **Simulation Time (`req_simt = 0.1 s`)**: The simulated duration is not set in `Deflection` but in the domain setup (`DomSetup`). A longer simulation time gives a finer frequency resolution in the postprocessing ($\Delta f = 1/T$), but it increases the computation time.
* **Time and Spatial Step (`dt = 0.5e-5 s`, `dx = 0.05 m`)**: The default step sizes satisfy the Courant–Friedrichs–Lewy (CFL) stability condition for the coupled multi-DOF track model and generally do not need to be changed.

Here we run the simulation for the vertical excitation and record the response at the excitation point.

In [3]:
from rolland.deflection import Deflection

# Run the simulation and record the response at the excitation point
defl_vert = Deflection(
    discr=domain,  # Domain setup
    excit=exc_vert,  # Vertical excitation
    store="excit",  # Storage mode
)